# QC-Py-31 - Transformer Encoder pour le Trading Quantitatif> **Niveau** : Avance | **Pre-requis** : QC-Py-30, QC-Py-22 | **Temps** : 90 min> **Objectif** : Implementer et entrainer un Transformer encoder-only sur des> donnees reelles S&P 500 (11 ans, top 50 actions) avec detection de regimes> de marche et deploiement sur QuantConnect.

> **[REFERENCE QC Cloud]** Ce notebook demontre l'entrainement GPU d'un modele> Transformer encoder pour la prediction de rendements financiers avec mecanisme> de detection de regimes de marche. Le modele est ensuite integre dans un> algorithme QuantConnect via ObjectStore.>> **Mode d'emploi** : Ce notebook a **trois parties** :> 1. **Theorie** : Architecture Transformer encoder-only avec positional encoding> 2. **Entrainement GPU** : Donnees reelles yfinance, PyTorch CUDA, multi-head attention> 3. **Integration QC** : Export vers ObjectStore et algorithme de trading

---## Partie 1 : Architecture Transformer pour le Trading (20 min)

### Du LSTM au TransformerLe Transformer (Vaswani et al., 2017) traite les sequences differemment du LSTM :- **Parallelisme** : tous les pas de temps sont traites simultanement- **Attention multi-tete** : plusieurs patrons de dependance captures en parallele- **Positional encoding** : informations positionnelles injectees via sin/cos**Pourquoi un encoder-only ?**Dans le trading, on n'a pas besoin de generation sequentielle. L'encoder suffitpour extraire des representations riches a partir d'une fenetre de prix.**Architecture cible** :- Positional encoding (sinusoidal, d_model=64)- 4 couches Transformer encoder (nhead=4, dim_feedforward=256)- Couches fully-connected pour la prediction binaire- Detection de regime de marche (bull/bear/sideways)

### Positional Encoding et Multi-Head AttentionLe positional encoding injecte l'information de position dans les embeddings :```PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))```La multi-head attention calcule plusieurs attention heads en parallele :```MultiHead(Q, K, V) = Concat(head_1, ..., head_h) * W_ohead_i = Attention(Q * W_q, K * W_k, V * W_v)```Avec 4 tetes sur d_model=64, chaque tete a dim=16, permettant au modelede capturer differentes relations temporelles (tendance, momentum, mean-reversion).

---

## Partie 2 : Setup et Donnees Reelles (15 min)

In [ ]:
# Imports standardsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport warningswarnings.filterwarnings("ignore")import torchimport torch.nn as nnimport torch.nn.functional as Fprint(f"PyTorch version: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Device: {DEVICE}")

In [ ]:
# Telechargement S&P 500 via yfinanceimport yfinance as yffrom datetime import datetimeSP50_TICKERS = [    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "BRK-B", "LLY", "AVGO", "TSM",    "JPM", "V", "UNH", "XOM", "MA", "JNJ", "PG", "COST", "HD", "MRK",    "ABBV", "CRM", "AMD", "NFLX", "ORCL", "WMT", "BAC", "KO", "CVX", "ACN",    "PEP", "TMO", "ADBE", "MCD", "CSCO", "INTC", "PFE", "ABT", "CMCSA", "NKE",    "TXN", "VZ", "MRVL", "QCOM", "UPS", "RTX", "LIN", "NEE", "GS", "IBM"]END_DATE = datetime(2025, 1, 1)START_DATE = datetime(2014, 1, 1)print(f"Telechargement de {len(SP50_TICKERS)} actions S&P 500...")data = yf.download(SP50_TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)print(f"Shape: {data.shape}")print(f"Dates: {data.index[0].strftime('%Y-%m-%d')} au {data.index[-1].strftime('%Y-%m-%d')}")

### InterpretationLes donnees couvrent la meme periode que QC-Py-30 pour permettre unecomparaison equitable entre LSTM et Transformer.

In [ ]:
# Preparation des features techniques (meme pipeline que QC-Py-30)from sklearn.preprocessing import StandardScalerclose_prices = data["Close"].dropna(axis=1, how="all")valid_tickers = close_prices.columns.tolist()FEATURE_COLS = ["ret_1d", "ret_5d", "ret_20d", "sma20_ratio", "sma50_ratio",                "vol_20d", "vol_5d", "vol_ratio", "rsi_14"]SEQUENCE_LENGTH = 60HORIZON = 5features_list = []for ticker in valid_tickers:    try:        df = pd.DataFrame(index=close_prices.index)        df["ret_1d"] = close_prices[ticker].pct_change()        df["ret_5d"] = close_prices[ticker].pct_change(5)        df["ret_20d"] = close_prices[ticker].pct_change(20)        sma_20 = close_prices[ticker].rolling(20).mean()        sma_50 = close_prices[ticker].rolling(50).mean()        df["sma20_ratio"] = close_prices[ticker] / sma_20 - 1        df["sma50_ratio"] = close_prices[ticker] / sma_50 - 1        df["vol_20d"] = df["ret_1d"].rolling(20).std()        df["vol_5d"] = df["ret_1d"].rolling(5).std()        vol = data["Volume"][ticker] if ticker in data["Volume"].columns else None        if vol is not None:            df["vol_ratio"] = vol / vol.rolling(20).mean()        delta = close_prices[ticker].diff()        gain = delta.where(delta > 0, 0).rolling(14).mean()        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()        rs = gain / loss.replace(0, 1e-10)        df["rsi_14"] = 100 - (100 / (1 + rs))        df["ticker"] = ticker        features_list.append(df.dropna())    except Exception:        passall_features = pd.concat(features_list)print(f"Features shape: {all_features.shape}")

In [ ]:
# Normalisation et creation des sequencesscalers = {}scaled_data = {}for ticker in valid_tickers:    ticker_data = all_features[all_features["ticker"] == ticker].copy()    if len(ticker_data) < SEQUENCE_LENGTH + HORIZON:        continue    scaler = StandardScaler()    scaled = scaler.fit_transform(ticker_data[FEATURE_COLS].values)    scalers[ticker] = scaler    targets = ticker_data["ret_5d"].shift(-HORIZON).values    scaled_data[ticker] = {"features": scaled, "targets": targets, "dates": ticker_data.index}X_all, y_all, ticker_all, date_all = [], [], [], []for ticker, d in scaled_data.items():    for i in range(len(d["features"]) - SEQUENCE_LENGTH - HORIZON):        X_all.append(d["features"][i:i + SEQUENCE_LENGTH])        y_all.append(d["targets"][i + SEQUENCE_LENGTH])        ticker_all.append(ticker)        date_all.append(d["dates"][i + SEQUENCE_LENGTH])X_all = np.array(X_all, dtype=np.float32)y_all = np.array(y_all, dtype=np.float32)y_class = (y_all > 0).astype(np.float32)n = len(X_all)train_end = int(n * 0.70)val_end = int(n * 0.85)np.random.seed(42)train_idx = np.arange(train_end)np.random.shuffle(train_idx)X_train, y_train = X_all[train_idx], y_class[train_idx]X_val, y_val = X_all[train_end:val_end], y_class[train_end:val_end]X_test, y_test = X_all[val_end:], y_class[val_end:]X_train_t = torch.tensor(X_train).to(DEVICE)y_train_t = torch.tensor(y_train).unsqueeze(1).to(DEVICE)X_val_t = torch.tensor(X_val).to(DEVICE)y_val_t = torch.tensor(y_val).unsqueeze(1).to(DEVICE)X_test_t = torch.tensor(X_test).to(DEVICE)y_test_t = torch.tensor(y_test).unsqueeze(1).to(DEVICE)print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")print(f"Positifs: {y_class.mean():.2%}")

### InterpretationLe pipeline de donnees est identique a QC-Py-30 pour comparer objectivementles architectures LSTM et Transformer.

---

## Partie 3 : Architecture Transformer Encoder (20 min)

In [ ]:
# Modele Transformer Encoder pour le Tradingclass PositionalEncoding(nn.Module):    """Positional encoding sinusoidal."""    def __init__(self, d_model, max_len=500):        super().__init__()        pe = torch.zeros(max_len, d_model)        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))        pe[:, 0::2] = torch.sin(position * div_term)        pe[:, 1::2] = torch.cos(position * div_term)        self.register_buffer("pe", pe.unsqueeze(0))    def forward(self, x):        return x + self.pe[:, :x.size(1)]class TransformerTradingModel(nn.Module):    """Transformer encoder-only pour la prediction de trading."""    def __init__(self, input_dim=9, d_model=64, nhead=4, num_layers=4,                 dim_feedforward=256, dropout=0.2):        super().__init__()        self.input_proj = nn.Linear(input_dim, d_model)        self.pos_encoder = PositionalEncoding(d_model)        encoder_layer = nn.TransformerEncoderLayer(            d_model=d_model, nhead=nhead,            dim_feedforward=dim_feedforward,            dropout=dropout, batch_first=True,            activation="gelu"        )        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)        self.layer_norm = nn.LayerNorm(d_model)        self.classifier = nn.Sequential(            nn.Linear(d_model, 32), nn.GELU(), nn.Dropout(dropout),            nn.Linear(32, 1), nn.Sigmoid()        )    def forward(self, x):        x = self.input_proj(x)        x = self.pos_encoder(x)        x = self.transformer(x)        x = self.layer_norm(x)        pooled = x.mean(dim=1)        return self.classifier(pooled), xmodel = TransformerTradingModel(    input_dim=len(FEATURE_COLS), d_model=64, nhead=4,    num_layers=4, dim_feedforward=256, dropout=0.2).to(DEVICE)n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f"TransformerTradingModel")print(f"  d_model: 64")print(f"  Attention heads: 4")print(f"  Encoder layers: 4")print(f"  Feedforward dim: 256")print(f"  Parametres: {n_params:,}")print(f"  Device: {DEVICE}")

### InterpretationLe Transformer encoder a 4 couches avec 4 tetes d'attention. Le d_model=64est projete depuis les 9 features d'entree. Le pooling moyen agrege lesrepresentations temporelles avant la classification.**Comparaison LSTM vs Transformer** :- LSTM : dependances sequentielles, memoire a long terme via cell state- Transformer : attention globale, parallelisme, pas de biais recurrent- Les deux peuvent capturer des dependances longues, mais differemment

---

## Partie 4 : Entrainement GPU (20 min)

In [ ]:
# Configuration de l'entrainementimport torch.optim as optimfrom torch.utils.data import DataLoader, TensorDatasetimport timeBATCH_SIZE = 256N_EPOCHS = 50PATIENCE = 10LR = 5e-4train_dataset = TensorDataset(X_train_t, y_train_t)val_dataset = TensorDataset(X_val_t, y_val_t)train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)criterion = nn.BCELoss()optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)scheduler = optim.lr_scheduler.OneCycleLR(    optimizer, max_lr=LR, epochs=N_EPOCHS,    steps_per_epoch=len(train_loader), pct_start=0.3)print(f"Configuration:")print(f"  Batch size: {BATCH_SIZE}")print(f"  Epochs max: {N_EPOCHS}")print(f"  Patience: {PATIENCE}")print(f"  Learning rate: {LR}")print(f"  Optimizer: AdamW (weight_decay=1e-4)")print(f"  Scheduler: OneCycleLR")

In [ ]:
# Boucle d'entrainementtrain_losses, val_losses, train_accs, val_accs = [], [], [], []best_val_loss = float("inf")best_epoch = 0patience_counter = 0print(f"{'Epoch':>5} | {'Tr Loss':>8} | {'Tr Acc':>7} | {'Va Loss':>8} | {'Va Acc':>7} | {'LR':>10} | {'Time':>5}")print("-" * 70)for epoch in range(1, N_EPOCHS + 1):    start = time.time()    model.train()    running_loss = correct = total = 0    for X_batch, y_batch in train_loader:        optimizer.zero_grad()        pred, _ = model(X_batch)        loss = criterion(pred, y_batch)        loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        optimizer.step()        scheduler.step()        running_loss += loss.item() * len(X_batch)        correct += ((pred > 0.5) == y_batch).sum().item()        total += len(X_batch)    train_loss = running_loss / total    train_acc = correct / total    model.eval()    running_loss = correct = total = 0    with torch.no_grad():        for X_batch, y_batch in val_loader:            pred, _ = model(X_batch)            loss = criterion(pred, y_batch)            running_loss += loss.item() * len(X_batch)            correct += ((pred > 0.5) == y_batch).sum().item()            total += len(X_batch)    val_loss = running_loss / total    val_acc = correct / total    elapsed = time.time() - start    train_losses.append(train_loss)    val_losses.append(val_loss)    train_accs.append(train_acc)    val_accs.append(val_acc)    lr = optimizer.param_groups[0]["lr"]    marker = ""    if val_loss < best_val_loss:        best_val_loss = val_loss        best_epoch = epoch        patience_counter = 0        marker = " *"        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),                    "val_loss": val_loss, "val_acc": val_acc}, "best_transformer_model.pt")    else:        patience_counter += 1    if epoch % 5 == 0 or epoch == 1 or marker:        print(f"{epoch:5d} | {train_loss:8.4f} | {train_acc:6.2%} | {val_loss:8.4f} | {val_acc:6.2%} | {lr:10.2e} | {elapsed:4.1f}s{marker}")    if patience_counter >= PATIENCE:        print(f"\nEarly stopping a l'epoch {epoch} (best: {best_epoch})")        breakprint("-" * 70)print(f"Meilleur modele: epoch {best_epoch}, val_loss={best_val_loss:.4f}")

### InterpretationLe Transformer converge generalement plus vite que le LSTM grace au parallelisme.Le learning rate plus bas (5e-4 vs 1e-3) stabilise l'entrainement des couches d'attention.

In [ ]:
# Visualisation des courbes d'apprentissagefig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(train_losses, label="Train", alpha=0.8)axes[0].plot(val_losses, label="Validation", alpha=0.8)axes[0].axvline(best_epoch - 1, color="red", linestyle="--", alpha=0.5, label=f"Best (ep {best_epoch})")axes[0].set_xlabel("Epoch")axes[0].set_ylabel("Loss (BCE)")axes[0].set_title("Transformer - Loss")axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].plot(train_accs, label="Train", alpha=0.8)axes[1].plot(val_accs, label="Validation", alpha=0.8)axes[1].axvline(best_epoch - 1, color="red", linestyle="--", alpha=0.5)axes[1].set_xlabel("Epoch")axes[1].set_ylabel("Accuracy")axes[1].set_title("Transformer - Accuracy")axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.savefig("transformer_training_curves.png", dpi=150, bbox_inches="tight")plt.show()

### InterpretationLe Transformer atteint souvent un meilleur taux de convergence que le LSTM.L'absence de recurrence permet un meilleur flux de gradients.

---

## Partie 5 : Evaluation et Detection de Regime (15 min)

In [ ]:
# Evaluation sur le set de testcheckpoint = torch.load("best_transformer_model.pt", weights_only=False)model.load_state_dict(checkpoint["model_state_dict"])model.eval()print(f"Meilleur modele charge: epoch {checkpoint['epoch']}, val_acc={checkpoint['val_acc']:.4f}")from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_scorewith torch.no_grad():    pred_proba, encoder_out = model(X_test_t)    pred_proba = pred_proba.cpu().numpy().flatten()    pred_class = (pred_proba > 0.5).astype(int)    true_class = y_testacc = accuracy_score(true_class, pred_class)prec = precision_score(true_class, pred_class)rec = recall_score(true_class, pred_class)f1 = f1_score(true_class, pred_class)auc = roc_auc_score(true_class, pred_proba)print(f"\nResultats sur le set de test:")print(f"  Accuracy:  {acc:.4f}")print(f"  Precision: {prec:.4f}")print(f"  Recall:    {rec:.4f}")print(f"  F1 Score:  {f1:.4f}")print(f"  AUC-ROC:   {auc:.4f}")

### InterpretationLe Transformer capture les relations non-lineaires entre les features gracea la multi-head attention. Compare au LSTM, il peut identifier des patronsplus complexes.

In [ ]:
# Detection de regime de marche via clustering des representationsfrom sklearn.cluster import KMeansfrom sklearn.decomposition import PCAencoder_outputs = encoder_out.mean(dim=1).cpu().numpy()print(f"Encoder output shape: {encoder_outputs.shape}")kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)regimes = kmeans.fit_predict(encoder_outputs)pca = PCA(n_components=2)emb_2d = pca.fit_transform(encoder_outputs)regime_names = {0: "Bull", 1: "Sideways", 2: "Bear"}regime_rets = {}for r in range(3):    mask = regimes == r    avg_ret = y_all[val_end:][mask].mean()    regime_rets[r] = avg_ret    if avg_ret > 0.002:        regime_names[r] = "Bull"    elif avg_ret < -0.002:        regime_names[r] = "Bear"    else:        regime_names[r] = "Sideways"fig, axes = plt.subplots(1, 2, figsize=(16, 6))for r in range(3):    mask = regimes == r    axes[0].scatter(emb_2d[mask, 0], emb_2d[mask, 1], alpha=0.3, s=5,                    label=f"{regime_names[r]} ({mask.sum()})")axes[0].set_xlabel("PC1")axes[0].set_ylabel("PC2")axes[0].set_title("Regimes de Marche (PCA des representations Transformer)")axes[0].legend()axes[0].grid(True, alpha=0.3)regime_acc = {}for r in range(3):    mask = regimes == r    regime_acc[regime_names[r]] = (pred_class[mask] == true_class[mask]).mean()bars = axes[1].bar(regime_acc.keys(), regime_acc.values(), color=["green", "gray", "red"], alpha=0.7)axes[1].set_ylabel("Accuracy")axes[1].set_title("Accuracy par Regime de Marche")axes[1].grid(True, alpha=0.3, axis="y")for bar, v in zip(bars, regime_acc.values()):    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,                 f"{v:.2%}", ha="center", fontsize=10)plt.tight_layout()plt.savefig("transformer_regimes.png", dpi=150, bbox_inches="tight")plt.show()print("\nDistribution des regimes:")for r in range(3):    mask = regimes == r    print(f"  {regime_names[r]}: {mask.sum()} samples, avg_ret={regime_rets[r]:.4f}, acc={regime_acc[regime_names[r]]:.2%}")

### InterpretationLe Transformer encode implicitement des informations sur le regime de marche.Le clustering KMeans sur les representations apprises revele 3 regimes :- **Bull** : rendements positifs, le modele est bon pour predire- **Sideways** : rendements proches de zero, performance moindre- **Bear** : rendements negatifs, le modele detecte bien les baisses

In [ ]:
# Backtest simulethreshold = 0.55test_dates = date_all[val_end:]test_rets = y_all[val_end:]port_returns = []for date in sorted(set(test_dates)):    mask = [i for i, d in enumerate(test_dates) if d == date]    date_preds = pred_proba[mask]    date_rets = test_rets[mask]    long_m = date_preds > threshold    short_m = date_preds < (1 - threshold)    if long_m.sum() > 0 or short_m.sum() > 0:        lr = date_rets[long_m].mean() if long_m.sum() > 0 else 0        sr = -date_rets[short_m].mean() if short_m.sum() > 0 else 0        n_pos = long_m.sum() + short_m.sum()        port_ret = (lr * long_m.sum() + sr * short_m.sum()) / max(n_pos, 1)    else:        port_ret = 0    port_returns.append(port_ret)port_s = pd.Series(port_returns)cumulative = (1 + port_s).cumprod()total_return = cumulative.iloc[-1] / cumulative.iloc[0] - 1ann_return = (1 + total_return) ** (252 / len(port_s)) - 1 if len(port_s) > 0 else 0sharpe = port_s.mean() / port_s.std() * np.sqrt(252) if port_s.std() > 0 else 0max_dd = (cumulative / cumulative.cummax() - 1).min()win_rate = (port_s > 0).mean()print(f"\nBacktest Simule Transformer:")print(f"  Rendement total: {total_return:+.2%}")print(f"  Rendement annuel: {ann_return:+.2%}")print(f"  Sharpe Ratio: {sharpe:.3f}")print(f"  Max Drawdown: {max_dd:.2%}")print(f"  Win Rate: {win_rate:.2%}")

### InterpretationLe backtest simule applique les predictions Transformer avec un seuil de 0.55.Le Transformer peut capturer des relations plus complexes que le LSTM, ce quipeut se traduire par un meilleur Sharpe ratio.

---

## Partie 6 : Integration QuantConnect ObjectStore (10 min)

In [ ]:
# Export du modele pour QuantConnectimport jsonexport_data = {    "model_state_dict": {k: v.cpu().numpy().tolist() for k, v in model.state_dict().items()},    "config": {        "model_type": "transformer_encoder",        "input_dim": len(FEATURE_COLS),        "d_model": 64,        "nhead": 4,        "num_layers": 4,        "dim_feedforward": 256,        "dropout": 0.2,        "feature_cols": FEATURE_COLS,        "sequence_length": SEQUENCE_LENGTH,        "horizon": HORIZON,        "scaler_means": {t: s.mean_.tolist() for t, s in scalers.items()},        "scaler_scales": {t: s.scale_.tolist() for t, s in scalers.items()},    },    "training_results": {        "best_epoch": best_epoch,        "val_acc": float(checkpoint['val_acc']),        "val_loss": float(checkpoint['val_loss']),        "test_accuracy": float(acc),        "test_auc": float(auc),        "test_sharpe": float(sharpe),    }}with open("transformer_trading_model.json", "w") as f:    json.dump(export_data, f)print(f"Modele exporte: transformer_trading_model.json")print(f"  Taille: {len(json.dumps(export_data)) / 1e6:.1f} MB")print(f"  Config: d_model=64, heads=4, layers=4")print(f"  Test AUC: {auc:.4f}")print(f"  Test Sharpe: {sharpe:.3f}")

### InterpretationLe modele Transformer est exporte au meme format que le LSTM pour compatibilite.La config inclut les hyperparametres specifiques au Transformer.

In [ ]:
# Template QuantConnect pour le Transformerqc_template = """# region importsfrom AlgorithmImports import *import json, numpy as np, torch, torch.nn as nn# endregionclass TransformerTradingAlgorithm(QCAlgorithm):    def Initialize(self):        self.SetStartDate(2023, 1, 1)        self.SetCash(100000)        self.sequence_length = 60        self.threshold = 0.55        self.symbols = [self.AddEquity(t, Resolution.Daily).Symbol            for t in ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL',                      'META', 'JPM', 'V', 'UNH', 'XOM']]        self.LoadModel()        self.Schedule.On(self.DateRules.EveryDay(),            self.TimeRules.AfterMarketOpen('SPY', 30), self.Rebalance)    def LoadModel(self):        if self.ObjectStore.ContainsKey('transformer_trading_model.json'):            data = json.loads(self.ObjectStore.Read('transformer_trading_model.json'))            self.model = self.BuildTransformer(data['config'])            state_dict = {k: torch.tensor(v) for k, v in data['model_state_dict'].items()}            self.model.load_state_dict(state_dict)            self.model.eval()    def BuildTransformer(self, cfg):        class PosEnc(nn.Module):            def __init__(self, d, ml=500):                super().__init__()                pe = torch.zeros(ml, d)                pos = torch.arange(0, ml, dtype=torch.float).unsqueeze(1)                div = torch.exp(torch.arange(0, d, 2).float() * (-np.log(10000.0) / d))                pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)                self.register_buffer("pe", pe.unsqueeze(0))            def forward(self, x): return x + self.pe[:, :x.size(1)]        class TransModel(nn.Module):            def __init__(self, i, d, nh, nl, ff, dp):                super().__init__()                self.proj = nn.Linear(i, d); self.pe = PosEnc(d)                enc = nn.TransformerEncoderLayer(d, nh, ff, dp, batch_first=True, activation="gelu")                self.enc = nn.TransformerEncoder(enc, nl)                self.cls = nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Dropout(dp), nn.Linear(32, 1), nn.Sigmoid())            def forward(self, x):                x = self.pe(self.proj(x)); x = self.enc(x).mean(dim=1); return self.cls(x)        return TransModel(cfg['input_dim'], cfg['d_model'], cfg['nhead'],                          cfg['num_layers'], cfg['dim_feedforward'], cfg['dropout'])    def Rebalance(self):        for symbol in self.symbols:            features = self.ComputeFeatures(symbol)            if features is None or len(features) < self.sequence_length:                continue            seq = torch.tensor(features[-self.sequence_length:]).unsqueeze(0).float()            with torch.no_grad(): prob = self.model(seq).item()            if prob > self.threshold:                self.SetHoldings(symbol, min(prob, 0.95) / len(self.symbols) * 2)            elif prob < (1 - self.threshold):                self.SetHoldings(symbol, 0)    def ComputeFeatures(self, symbol):        hist = self.History(symbol, 70, Resolution.Daily)        if hist.empty: return None        close = hist['close'].values        ret = np.diff(close) / close[:-1]        return np.column_stack([ret[-60:]])"""print("Template QC Transformer genere.")print("Pour deployer: uploader le JSON dans ObjectStore + copier le template.")

### InterpretationLe template QC charge le Transformer depuis ObjectStore. La methode `Rebalance`est identique a celle du LSTM, mais le modele interne est un Transformer encoder.**Avantage du Transformer en production** : inference plus rapide grace auparallelisme natif, pas de traitement sequentiel.

---

## Conclusion et Perspectives### Ce que nous avons construit1. **Transformer Encoder** : 4 couches avec 4 tetes d'attention (d_model=64)   et positional encoding sinusoidal2. **Detection de regimes** : clustering des representations pour identifier   bull/bear/sideways implicitement3. **Comparaison LSTM vs Transformer** : meme pipeline de donnees pour evaluation   equitable des deux architectures4. **Integration QC** : export ObjectStore + algorithme de deploiement### LSTM vs Transformer : quand choisir quoi ?| Critere | LSTM | Transformer ||---------|------|-------------|| Longueur sequence | Bon pour sequences longues | Bon pour sequences moyennes || Parallelisme | Sequentiel | Pleinement parallele || Donnees requises | Moins | Plus || Interpretation | Attention LSTM | Multi-head attention + regimes || Inference | Plus lent | Plus rapide |### Prochaine etapeLe notebook **QC-Py-32** implementera un agent RL (DQN) qui apprendune politique de trading par interaction avec un environnement de marche.

In [ ]:
# Resume finalprint("=" * 60)print("RESUME - QC-Py-31 Transformer Training")print("=" * 60)print(f"Architecture:     Transformer Encoder, d=64, h=4, L=4")print(f"Donnees:          {len(valid_tickers)} actions S&P 500, 11 ans")print(f"Features:         {len(FEATURE_COLS)} techniques")print(f"Sequence length:  {SEQUENCE_LENGTH} jours")print(f"Parametres:       {n_params:,}")print(f"Meilleur epoch:   {best_epoch}")print(f"Test Accuracy:    {acc:.4f}")print(f"Test AUC-ROC:     {auc:.4f}")print(f"Test Sharpe:      {sharpe:.3f}")print(f"Regimes detectes: {len(set(regimes))} (Bull/Sideways/Bear)")print("=" * 60)